In [1]:
import os
import json
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
with open("data_split.json", "r") as f:
    split_data = json.load(f)

train_subjects = split_data["train"]
val_subjects = split_data["validation"]
test_subjects = split_data["test"]

print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))

Train: 1000
Validation: 125
Test: 126


In [3]:
train_data = r"Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

In [4]:
def preprocess_t2f(image):
    # Expected original BraTS shape
    if image.shape != (240, 240, 155):
        raise ValueError(f"Unexpected image shape: {image.shape}")

    image = image[16:224, 8:232, :]

    image = np.pad(
        image,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    foreground = image > 0

    if not np.any(foreground):
        raise ValueError("No foreground voxels found")

    upper = np.percentile(image[foreground], 99.9)

    image = np.clip(image, 0, upper)

    image = image / upper

    image[~foreground] = 0

    return image.astype(np.float32)

In [5]:
def preprocess_mask(mask):
    # Expected original BraTS shape
    if mask.shape != (240, 240, 155):
        raise ValueError(f"Unexpected mask shape: {mask.shape}")

    # Apply exactly the same spatial crop as T2f
    mask = mask[16:224, 8:232, :]

    # Apply exactly the same z-padding as T2f
    mask = np.pad(
        mask,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    return mask.astype(np.int64)

In [6]:
def calculate_tumour_entropy(image, mask, num_bins=256):
    # Whole tumour = all non-zero tumour labels
    tumour_region = mask > 0

    if not np.any(tumour_region):
        raise ValueError("No tumour voxels found")

    tumour_values = image[tumour_region]

    # T2f has already been normalized to [0, 1]
    tumour_values = np.clip(tumour_values, 0.0, 1.0)

    hist, _ = np.histogram(
        tumour_values,
        bins=num_bins,
        range=(0.0, 1.0),
        density=False
    )

    probabilities = hist.astype(np.float64)
    probabilities = probabilities / probabilities.sum()

    probabilities = probabilities[probabilities > 0]

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    return np.float32(entropy)

In [7]:
from torch.utils.data import Dataset, DataLoader

class BraTSDataset(Dataset):
    def __init__(self, subjects, data_dir):
        self.subjects = subjects
        self.data_dir = data_dir

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        subject = self.subjects[idx]
        subject_path = os.path.join(self.data_dir, subject)

        files = os.listdir(subject_path)

        t2f_file = [f for f in files if "t2f" in f.lower()][0]
        seg_file = [f for f in files if "seg" in f.lower()][0]

        # Load T2f
        image = nib.load(
            os.path.join(subject_path, t2f_file)
        ).get_fdata()

        # Load segmentation
        mask = nib.load(
            os.path.join(subject_path, seg_file)
        ).get_fdata()

        # Apply preprocessing
        image = preprocess_t2f(image)
        mask = preprocess_mask(mask)

        # Calculate whole-tumour Shannon entropy
        entropy = calculate_tumour_entropy(
            image,
            mask
        )

        # Convert to tensors
        image = torch.from_numpy(
            image
        ).float().unsqueeze(0)

        mask = torch.from_numpy(
            mask
        ).long().unsqueeze(0)

        entropy = torch.tensor(
            entropy,
            dtype=torch.float32
        )

        return {
            "image": image,
            "mask": mask,
            "heterogeneity": entropy,
            "subject": subject
        }

In [8]:
train_dataset = BraTSDataset(
    subjects=train_subjects,
    data_dir=train_data
)

print("Dataset size:", len(train_dataset))

Dataset size: 1000


In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

In [10]:
class VAEBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),
            nn.SiLU(),

            nn.Conv3d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),
            nn.SiLU()
        )

    def forward(self, x):
        return self.block(x)

In [11]:
# ============================================================
# Conditional LDM V6
# Higher-capacity x4 / 8-channel VAE Encoder
# ============================================================

class VAEEncoder3D(nn.Module):

    def __init__(
        self,
        in_channels=1,
        base_channels=32,
        latent_channels=8
    ):
        super().__init__()

        # [B, 1, 208, 224, 160]
        # ->
        # [B, 32, 208, 224, 160]

        self.enc1 = VAEBlock3D(
            in_channels,
            base_channels
        )

        # /2
        # [B, 32, 208, 224, 160]
        # ->
        # [B, 64, 104, 112, 80]

        self.down1 = nn.Conv3d(
            base_channels,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.enc2 = VAEBlock3D(
            base_channels * 2,
            base_channels * 2
        )

        # /4
        # [B, 64, 104, 112, 80]
        # ->
        # [B, 128, 52, 56, 40]

        self.down2 = nn.Conv3d(
            base_channels * 2,
            base_channels * 4,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.bottleneck = VAEBlock3D(
            base_channels * 4,
            base_channels * 4
        )

        # 128 feature channels -> 8 latent channels

        self.to_mu = nn.Conv3d(
            base_channels * 4,
            latent_channels,
            kernel_size=1
        )

        self.to_logvar = nn.Conv3d(
            base_channels * 4,
            latent_channels,
            kernel_size=1
        )


    def forward(self, x):

        x = self.enc1(x)

        x = self.down1(x)
        x = self.enc2(x)

        x = self.down2(x)
        x = self.bottleneck(x)

        mu = self.to_mu(x)
        logvar = self.to_logvar(x)

        return mu, logvar

In [12]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

In [13]:
# ============================================================
# Conditional LDM V6
# Higher-capacity x4 / 8-channel VAE Decoder
# ============================================================

class VAEDecoder3D(nn.Module):

    def __init__(
        self,
        out_channels=1,
        base_channels=32,
        latent_channels=8
    ):
        super().__init__()

        # [B, 8, 52, 56, 40]
        # ->
        # [B, 128, 52, 56, 40]

        self.from_latent = nn.Conv3d(
            latent_channels,
            base_channels * 4,
            kernel_size=3,
            padding=1
        )

        self.dec2 = VAEBlock3D(
            base_channels * 4,
            base_channels * 4
        )

        # ×2
        # 128 -> 64
        # 52×56×40 -> 104×112×80

        self.up2 = nn.ConvTranspose3d(
            base_channels * 4,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.dec1 = VAEBlock3D(
            base_channels * 2,
            base_channels * 2
        )

        # ×2
        # 64 -> 32
        # 104×112×80 -> 208×224×160

        self.up1 = nn.ConvTranspose3d(
            base_channels * 2,
            base_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.final_block = VAEBlock3D(
            base_channels,
            base_channels
        )

        self.output_conv = nn.Conv3d(
            base_channels,
            out_channels,
            kernel_size=1
        )


    def forward(self, z):

        x = self.from_latent(z)

        x = self.dec2(x)

        x = self.up2(x)

        x = self.dec1(x)

        x = self.up1(x)

        x = self.final_block(x)

        x = self.output_conv(x)

        x = torch.sigmoid(x)

        return x

In [14]:
# ============================================================
# Conditional LDM V6
# x4 / 8-channel / base32 VAE
# ============================================================

class VAE3D(nn.Module):

    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_channels=32,
        latent_channels=8
    ):
        super().__init__()

        self.encoder = VAEEncoder3D(
            in_channels=in_channels,
            base_channels=base_channels,
            latent_channels=latent_channels
        )

        self.decoder = VAEDecoder3D(
            out_channels=out_channels,
            base_channels=base_channels,
            latent_channels=latent_channels
        )


    def forward(self, x):

        mu, logvar = self.encoder(x)

        z = reparameterize(
            mu,
            logvar
        )

        reconstruction = self.decoder(z)

        return (
            reconstruction,
            mu,
            logvar,
            z
        )

In [15]:
# ============================================================
# Conditional LDM V6
# VAE loss
#
# L1 reconstruction
# + multi-scale reconstruction
# + 3D gradient/detail preservation
# + KL regularisation
# ============================================================


def gradient_3d_loss_v6(
    reconstruction,
    target
):

    # X direction
    recon_dx = (
        reconstruction[:, :, 1:, :, :]
        -
        reconstruction[:, :, :-1, :, :]
    )

    target_dx = (
        target[:, :, 1:, :, :]
        -
        target[:, :, :-1, :, :]
    )


    # Y direction
    recon_dy = (
        reconstruction[:, :, :, 1:, :]
        -
        reconstruction[:, :, :, :-1, :]
    )

    target_dy = (
        target[:, :, :, 1:, :]
        -
        target[:, :, :, :-1, :]
    )


    # Z direction
    recon_dz = (
        reconstruction[:, :, :, :, 1:]
        -
        reconstruction[:, :, :, :, :-1]
    )

    target_dz = (
        target[:, :, :, :, 1:]
        -
        target[:, :, :, :, :-1]
    )


    loss_x = F.l1_loss(
        recon_dx,
        target_dx
    )

    loss_y = F.l1_loss(
        recon_dy,
        target_dy
    )

    loss_z = F.l1_loss(
        recon_dz,
        target_dz
    )


    gradient_loss = (
        loss_x
        + loss_y
        + loss_z
    ) / 3.0


    return gradient_loss



def vae_loss_v6(
    reconstruction,
    target,
    mu,
    logvar,
    kl_weight=1e-6,
    multiscale_weight=0.1,
    gradient_weight=0.1
):

    # ========================================================
    # Full-resolution L1
    # ========================================================

    recon_loss = F.l1_loss(
        reconstruction,
        target
    )


    # ========================================================
    # Half-resolution structural loss
    # ========================================================

    recon_half = F.avg_pool3d(
        reconstruction,
        kernel_size=2,
        stride=2
    )

    target_half = F.avg_pool3d(
        target,
        kernel_size=2,
        stride=2
    )


    # ========================================================
    # Quarter-resolution structural loss
    # ========================================================

    recon_quarter = F.avg_pool3d(
        reconstruction,
        kernel_size=4,
        stride=4
    )

    target_quarter = F.avg_pool3d(
        target,
        kernel_size=4,
        stride=4
    )


    half_loss = F.l1_loss(
        recon_half,
        target_half
    )

    quarter_loss = F.l1_loss(
        recon_quarter,
        target_quarter
    )


    multiscale_loss = (
        0.5 * half_loss
        +
        0.5 * quarter_loss
    )


    # ========================================================
    # High-frequency / anatomical edge loss
    # ========================================================

    gradient_loss = gradient_3d_loss_v6(
        reconstruction,
        target
    )


    # ========================================================
    # KL divergence
    # ========================================================

    kl_loss = -0.5 * torch.mean(
        1
        + logvar
        - mu.pow(2)
        - logvar.exp()
    )


    # ========================================================
    # Total
    # ========================================================

    total_loss = (
        recon_loss
        +
        multiscale_weight
        * multiscale_loss
        +
        gradient_weight
        * gradient_loss
        +
        kl_weight
        * kl_loss
    )


    return (
        total_loss,
        recon_loss,
        kl_loss,
        multiscale_loss,
        gradient_loss
    )

In [16]:
# ============================================================
# Conditional LDM V6
# VAE training
# ============================================================


def train_vae_v6(
    model,
    train_loader,
    epochs,
    optimizer,
    device,

    checkpoint_dir="vae_x4_8ch_v6_checkpoints",

    kl_weight=1e-6,
    multiscale_weight=0.1,
    gradient_weight=0.1,

    start_epoch=0
):

    os.makedirs(
        checkpoint_dir,
        exist_ok=True
    )


    loss_history = []
    recon_history = []
    kl_history = []
    multiscale_history = []
    gradient_history = []


    use_amp = (
        device.type == "cuda"
    )


    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )


    for local_epoch in range(
        epochs
    ):

        current_epoch = (
            start_epoch
            + local_epoch
            + 1
        )


        model.train()


        epoch_loss = 0.0
        epoch_recon = 0.0
        epoch_kl = 0.0
        epoch_multiscale = 0.0
        epoch_gradient = 0.0


        for batch_idx, batch in enumerate(
            train_loader
        ):

            x = batch["image"].to(
                device,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.amp.autocast(
                device_type="cuda",
                enabled=use_amp
            ):

                (
                    reconstruction,
                    mu,
                    logvar,
                    z
                ) = model(x)


                (
                    loss,
                    recon_loss,
                    kl_loss,
                    multiscale_loss,
                    gradient_loss
                ) = vae_loss_v6(
                    reconstruction=reconstruction,
                    target=x,
                    mu=mu,
                    logvar=logvar,

                    kl_weight=kl_weight,

                    multiscale_weight=(
                        multiscale_weight
                    ),

                    gradient_weight=(
                        gradient_weight
                    )
                )


            scaler.scale(
                loss
            ).backward()


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )


            scaler.step(
                optimizer
            )

            scaler.update()


            epoch_loss += loss.item()
            epoch_recon += recon_loss.item()
            epoch_kl += kl_loss.item()

            epoch_multiscale += (
                multiscale_loss.item()
            )

            epoch_gradient += (
                gradient_loss.item()
            )


            if (
                batch_idx + 1
            ) % 10 == 0:

                print(
                    f"Epoch {current_epoch}/{start_epoch + epochs} | "
                    f"Batch {batch_idx + 1}/{len(train_loader)} | "
                    f"Loss={loss.item():.6f} | "
                    f"Recon={recon_loss.item():.6f} | "
                    f"KL={kl_loss.item():.6f} | "
                    f"Multi={multiscale_loss.item():.6f} | "
                    f"Grad={gradient_loss.item():.6f}"
                )


        n = len(
            train_loader
        )


        avg_loss = (
            epoch_loss / n
        )

        avg_recon = (
            epoch_recon / n
        )

        avg_kl = (
            epoch_kl / n
        )

        avg_multiscale = (
            epoch_multiscale / n
        )

        avg_gradient = (
            epoch_gradient / n
        )


        loss_history.append(
            avg_loss
        )

        recon_history.append(
            avg_recon
        )

        kl_history.append(
            avg_kl
        )

        multiscale_history.append(
            avg_multiscale
        )

        gradient_history.append(
            avg_gradient
        )


        print()
        print(
            f"Epoch {current_epoch} completed | "
            f"Loss={avg_loss:.6f} | "
            f"Recon={avg_recon:.6f} | "
            f"KL={avg_kl:.6f} | "
            f"Multi={avg_multiscale:.6f} | "
            f"Grad={avg_gradient:.6f}"
        )
        print()


        checkpoint_path = os.path.join(
            checkpoint_dir,
            (
                f"vae_v6_epoch_"
                f"{current_epoch:03d}.pt"
            )
        )


        torch.save(
            {
                "epoch": current_epoch,

                "model_state_dict": (
                    model.state_dict()
                ),

                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),

                "loss": avg_loss,

                "recon_loss": avg_recon,

                "kl_loss": avg_kl,

                "multiscale_loss": (
                    avg_multiscale
                ),

                "gradient_loss": (
                    avg_gradient
                ),

                "kl_weight": (
                    kl_weight
                ),

                "multiscale_weight": (
                    multiscale_weight
                ),

                "gradient_weight": (
                    gradient_weight
                ),

                "base_channels": 32,

                "latent_channels": 8
            },
            checkpoint_path
        )


        print(
            "Saved:",
            checkpoint_path
        )


        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_v6_loss_history.npy"
            ),
            np.asarray(
                loss_history,
                dtype=np.float32
            )
        )


        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_v6_recon_history.npy"
            ),
            np.asarray(
                recon_history,
                dtype=np.float32
            )
        )


        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_v6_kl_history.npy"
            ),
            np.asarray(
                kl_history,
                dtype=np.float32
            )
        )


        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_v6_multiscale_history.npy"
            ),
            np.asarray(
                multiscale_history,
                dtype=np.float32
            )
        )


        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_v6_gradient_history.npy"
            ),
            np.asarray(
                gradient_history,
                dtype=np.float32
            )
        )


    return (
        loss_history,
        recon_history,
        kl_history,
        multiscale_history,
        gradient_history
    )

In [15]:
def load_vae_checkpoint(
    model,
    optimizer,
    path,
    device
):
    checkpoint = torch.load(
        path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    if optimizer is not None:
        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

    loaded_epoch = checkpoint["epoch"]

    print(
        f"Loaded VAE checkpoint from epoch {loaded_epoch}"
    )

    return loaded_epoch

In [18]:
@torch.no_grad()
def reconstruct_vae(
    model,
    image,
    device
):
    model.eval()

    image = image.to(device)

    reconstruction, mu, logvar, z = model(image)

    return reconstruction

In [16]:
# ============================================================
# Conditional LDM V6
# Initialise and train the new x4 / 8-channel VAE
# ============================================================


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


vae = VAE3D(
    in_channels=1,
    out_channels=1,
    base_channels=32,
    latent_channels=8
).to(device)


optimizer_vae = torch.optim.AdamW(
    vae.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


total_parameters = sum(
    p.numel()
    for p in vae.parameters()
)


trainable_parameters = sum(
    p.numel()
    for p in vae.parameters()
    if p.requires_grad
)


print(
    "Device:",
    device
)

print(
    "V6 VAE parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

print(
    "Spatial compression:",
    "4 x 4 x 4"
)

print(
    "Latent channels:",
    8
)

print(
    "Expected latent shape:",
    "[B, 8, 52, 56, 40]"
)


# ============================================================
# Train V6 VAE from scratch
# ============================================================

vae_history = train_vae_v6(
    model=vae,
    train_loader=train_loader,

    epochs=200,

    optimizer=optimizer_vae,

    device=device,

    checkpoint_dir=(
        "vae_x4_8ch_v6_checkpoints"
    ),

    kl_weight=1e-6,

    multiscale_weight=0.1,

    gradient_weight=0.1,

    start_epoch=0
)


# ============================================================
# Freeze VAE after training
# ============================================================

vae.eval()

for p in vae.parameters():
    p.requires_grad = False


print()
print(
    "V6 VAE training complete."
)

print(
    "VAE frozen:",
    all(
        not p.requires_grad
        for p in vae.parameters()
    )
)

Device: cuda
Loaded frozen VAE epoch: 15
VAE frozen: True


In [17]:
# ============================================================
# Conditional LDM V6
# Compute 8-channel latent statistics
# ============================================================


latent_stats_path = (
    "conditional_ldm_v6_latent_stats.npz"
)


LATENT_CHANNELS = 8


if os.path.exists(
    latent_stats_path
):

    stats = np.load(
        latent_stats_path
    )


    LATENT_MEAN = torch.tensor(
        stats["mean"],
        dtype=torch.float32
    ).view(
        1,
        LATENT_CHANNELS,
        1,
        1,
        1
    )


    LATENT_STD = torch.tensor(
        stats["std"],
        dtype=torch.float32
    ).view(
        1,
        LATENT_CHANNELS,
        1,
        1,
        1
    )


    print(
        "Loaded cached V6 "
        "latent statistics."
    )


else:

    print(
        "Computing V6 "
        "8-channel latent statistics..."
    )


    channel_sum = torch.zeros(
        LATENT_CHANNELS,
        dtype=torch.float64
    )


    channel_sq_sum = torch.zeros(
        LATENT_CHANNELS,
        dtype=torch.float64
    )


    voxel_count = 0


    vae.eval()


    with torch.no_grad():

        for i in range(
            len(train_dataset)
        ):

            x = (
                train_dataset[i]["image"]
                .unsqueeze(0)
                .to(device)
            )


            mu, logvar = (
                vae.encoder(x)
            )


            # Deterministic latent
            z = mu


            assert (
                z.shape[1]
                == LATENT_CHANNELS
            ), (
                f"Expected "
                f"{LATENT_CHANNELS} "
                f"latent channels, "
                f"got {z.shape}"
            )


            z_cpu = (
                z.detach()
                .cpu()
                .double()
            )


            channel_sum += (
                z_cpu.sum(
                    dim=(0, 2, 3, 4)
                )
            )


            channel_sq_sum += (
                z_cpu ** 2
            ).sum(
                dim=(0, 2, 3, 4)
            )


            voxel_count += (
                z.shape[0]
                * z.shape[2]
                * z.shape[3]
                * z.shape[4]
            )


            if (
                i + 1
            ) % 100 == 0:

                print(
                    f"Processed "
                    f"{i + 1}/"
                    f"{len(train_dataset)}"
                )


    mean = (
        channel_sum
        / voxel_count
    )


    variance = (
        channel_sq_sum
        / voxel_count
        -
        mean ** 2
    )


    std = torch.sqrt(
        torch.clamp(
            variance,
            min=1e-12
        )
    )


    LATENT_MEAN = (
        mean.float()
        .view(
            1,
            LATENT_CHANNELS,
            1,
            1,
            1
        )
    )


    LATENT_STD = (
        std.float()
        .view(
            1,
            LATENT_CHANNELS,
            1,
            1,
            1
        )
    )


    np.savez(
        latent_stats_path,
        mean=mean.numpy(),
        std=std.numpy()
    )


    print(
        "Saved:",
        latent_stats_path
    )


print()
print(
    "LATENT_MEAN:"
)

print(
    LATENT_MEAN.flatten()
)


print()
print(
    "LATENT_STD:"
)

print(
    LATENT_STD.flatten()
)


print()
print(
    "Latent statistics shape:",
    LATENT_MEAN.shape
)


assert (
    LATENT_MEAN.shape
    ==
    (
        1,
        8,
        1,
        1,
        1
    )
)


assert torch.all(
    LATENT_STD > 0
)

Loaded cached latent statistics.
LATENT_MEAN: tensor([-0.0313, -0.1939,  0.1352,  0.0145])
LATENT_STD: tensor([0.6804, 0.9484, 1.4222, 0.2118])


In [18]:
entropy_stats_path = (
    "conditional_ldm_v4_entropy_stats.npz"
)

if os.path.exists(
    entropy_stats_path
):

    stats = np.load(
        entropy_stats_path
    )

    ENTROPY_MEAN = float(
        stats["mean"]
    )

    ENTROPY_STD = float(
        stats["std"]
    )

    print(
        "Loaded cached entropy statistics."
    )

else:

    print(
        "Computing entropy statistics..."
    )

    entropy_values = []

    for i in range(
        len(train_dataset)
    ):

        entropy_values.append(
            train_dataset[i][
                "heterogeneity"
            ].item()
        )

    entropy_values = np.asarray(
        entropy_values,
        dtype=np.float32
    )

    ENTROPY_MEAN = float(
        entropy_values.mean()
    )

    ENTROPY_STD = float(
        entropy_values.std()
    )

    np.savez(
        entropy_stats_path,
        mean=ENTROPY_MEAN,
        std=ENTROPY_STD
    )


print(
    "Entropy mean:",
    ENTROPY_MEAN
)

print(
    "Entropy std:",
    ENTROPY_STD
)

assert ENTROPY_STD > 0

Loaded cached entropy statistics.
Entropy mean: 6.870205879211426
Entropy std: 0.33203670382499695
